# LC 684 — Redundant Connection
**Difficulty:** Medium &nbsp;|&nbsp; **Category:** Graphs
**Pattern:** Union-Find — Detect the Edge that
Creates a Cycle

<div style="border-left:4px solid purple; padding:10px 16px;
            background:#f5f0ff; margin-top:12px;">
<strong>Core Insight:</strong> Process edges one
by one. Use Union-Find to track connected
components. The first edge whose two endpoints
are already in the same component creates a cycle
— that is the redundant edge.
</div>

## Official Problem Statement

You are given a graph that started as a tree with
`n` nodes (labeled 1 to n) and one additional
edge. The graph is represented as an array
`edges` where `edges[i] = [ai, bi]` indicates
there is an edge between `ai` and `bi`.

Return an edge that can be removed so that the
resulting graph is a tree. If there are multiple
answers, return the one that occurs last in the
input.

**Example 1:**
```
Input:  edges = [[1,2],[1,3],[2,3]]
Output: [2,3]
```
**Example 2:**
```
Input:  edges = [[1,2],[2,3],[3,4],[1,4],[1,5]]
Output: [1,4]
```

**Constraints:**
- `n == edges.length`
- `3 <= n <= 1000`
- `1 <= ai, bi <= n`
- `ai != bi`
- No repeated edges

## What This Is Actually Asking

You have a tree (connected graph with no cycles)
with exactly one extra edge added. Find that extra
edge — the one that creates the cycle. If multiple
edges could be the answer, return the last one
listed.

## Walk Through an Example by Hand

```
edges = [[1,2],[2,3],[3,4],[1,4],[1,5]]

Union-Find: parent = [0,1,2,3,4,5]  (1-indexed)

Edge [1,2]: find(1)=1  find(2)=2  different
  union: parent[2]=1   components: {1,2} {3} {4} {5}

Edge [2,3]: find(2)=1  find(3)=3  different
  union: parent[3]=1   components: {1,2,3} {4} {5}

Edge [3,4]: find(3)=1  find(4)=4  different
  union: parent[4]=1   components: {1,2,3,4} {5}

Edge [1,4]: find(1)=1  find(4)=1  SAME! -> cycle!
  Return [1,4]

(edge [1,5] not reached — [1,4] is the last
 redundant edge found in input order)
```

## The Picture

```
Tree (no extra edge):    After adding [1,4]:
      1                        1
     /|\                      /|\  \
    2 3 5                    2 3 5  \
    |                        |      |
    ... 4                    4 <----+
                             (cycle!)

Union-Find: two operations

  find(x): walk parent pointers to root
           (path compression: point directly to root)

  union(x, y):
    rx = find(x)  ry = find(y)
    if rx == ry: CYCLE -> return this edge
    else: parent[rx] = ry  (merge components)

  Path compression + union by rank -> near O(1) each op
```

## When To Use This Pattern

- When asked to **detect the edge that creates a
  cycle**, think **Union-Find — find before union**
- When `find(a) == find(b)` before union, think
  **same component — this edge is redundant**
- When the problem says "tree + one extra edge",
  think **exactly one cycle — first cycle edge wins**
- When components must merge efficiently, think
  **union by rank + path compression — O(α) ≈ O(1)**

## The Approach

Initialise a parent array where each node is its
own parent. For each edge, find the root of both
endpoints using path compression. If both roots
are the same, a cycle is created — return this
edge. Otherwise merge the two components by
pointing one root to the other.

In [ ]:
from typing import List  # type hints for the solution

In [ ]:
def test_harness(func):
    tests = [
        # (edges, expected)
        ([[1,2],[1,3],[2,3]],             [2,3]),
        ([[1,2],[2,3],[3,4],[1,4],[1,5]], [1,4]),
        ([[1,2],[2,3],[1,3]],             [1,3]),
        ([[1,2],[2,3],[3,1]],             [3,1]),  # last in list
        ([[1,4],[3,4],[1,3],[1,2],[3,5]], [1,3]),
    ]

    passed = 0
    for i, (edges, expected) in enumerate(tests):
        result = func([e[:] for e in edges])
        ok = result == expected
        status = "PASSED" if ok else "FAILED"
        if ok:
            passed += 1
        print(
            f"Test {i+1}: {status} | "
            f"edges={edges} | "
            f"expected={expected} | got={result}"
        )

    print(f"\n{passed}/{len(tests)} tests passed")

In [ ]:
def findRedundantConnection(
    edges: List[List[int]]
) -> List[int]:
    """
    Return the redundant edge that creates a cycle.

    Union-Find with path compression. For each edge
    [a,b]: if find(a)==find(b) return the edge (cycle).
    Otherwise union by pointing root of a to root of b.

    Time:  O(n * α(n)) ≈ O(n) — nearly linear
    Space: O(n) — parent array
    """
    pass


# Quick debug — run this cell while building
print(findRedundantConnection([[1,2],[1,3],[2,3]]))
# [2,3]
print(findRedundantConnection(
    [[1,2],[2,3],[3,4],[1,4],[1,5]]
))
# [1,4]
print(findRedundantConnection([[1,2],[2,3],[3,1]]))
# [3,1]

In [ ]:
# Uncomment and run when solution is ready
# test_harness(findRedundantConnection)

## Complexity

| Approach | Time | Space |
|---|---|---|
| DFS cycle detection per edge | O(n²) | O(n) |
| Union-Find (basic) | O(n log n) | O(n) |
| Union-Find + path compression | O(n * α(n)) | O(n) |

α(n) is the inverse Ackermann function — for all
practical n it is at most 4. Path compression
makes Union-Find essentially O(1) per operation.

## Real World Connection

At Citi, the network topology is modelled as a
tree of switches and servers. When a cabling error
introduces a redundant link, it creates a network
loop that can cause broadcast storms.
The redundant connection detector identifies the
offending cable using Union-Find on the topology
graph — the same algorithm as this problem.
On AWS, VPC routing tables are similarly checked:
Union-Find on subnet peering connections flags
the redundant route that would create a routing
loop before it is activated.

> **Simplicity and clarity is Gold.** — Sean's Study Mantra